# Section 01: Designing Data Processing Systems

**CareerAlign GCP Professional Data Engineer**

This notebook covers:
1. IAM and access control patterns
2. Encryption options (GMEK, CMEK, CSEK)
3. PII detection and de-identification with Cloud DLP
4. Dataform SQLX transformation patterns
5. Data validation and quality checks
6. Data migration service comparisons

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project. Local-only cells run anywhere.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-pde/notebooks/01-designing-data-systems.ipynb)

---
## Setup & Installations

In [ ]:
# Install required packages
!pip install -q google-cloud-bigquery google-cloud-dlp google-cloud-kms pandas

In [ ]:
import pandas as pd
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

---
## 1. IAM and Access Control Patterns

This section demonstrates IAM role types, BigQuery row-level security, and column-level policy tags.

In [ ]:
# --- IAM Role Types Comparison ---

iam_roles = pd.DataFrame({
    'Role Type': ['Basic', 'Basic', 'Basic', 'Predefined', 'Predefined', 'Predefined', 'Custom'],
    'Role': ['roles/viewer', 'roles/editor', 'roles/owner', 
             'roles/bigquery.dataViewer', 'roles/bigquery.dataEditor', 
             'roles/bigquery.admin', 'roles/custom.dataAnalyst'],
    'Scope': ['All resources', 'All resources', 'All resources',
             'BigQuery datasets', 'BigQuery datasets', 
             'BigQuery project', 'Custom selection'],
    'Recommended': ['No - too broad', 'No - too broad', 'No - too broad',
                    'Yes', 'Yes', 'Use sparingly', 'Yes - most precise'],
})

print("=== IAM Role Types ===")
print(iam_roles.to_string(index=False))
print()
print("Key Principle: Always follow LEAST PRIVILEGE.")
print("Use predefined roles for standard access, custom roles for precise control.")

In [ ]:
# --- BigQuery Row-Level Security SQL Examples ---

row_level_sql = """
-- Create a row access policy: US analysts can only see US data
CREATE ROW ACCESS POLICY us_only
ON `project.dataset.sales`
GRANT TO ("group:us-analysts@example.com")
FILTER USING (region = 'US');

-- Create another policy: EU analysts see EU data
CREATE ROW ACCESS POLICY eu_only  
ON `project.dataset.sales`
GRANT TO ("group:eu-analysts@example.com")
FILTER USING (region = 'EU');

-- Admins see everything (TRUE filter)
CREATE ROW ACCESS POLICY admin_all
ON `project.dataset.sales`
GRANT TO ("group:data-admins@example.com")
FILTER USING (TRUE);

-- List all policies on a table
SELECT * FROM `project.dataset.INFORMATION_SCHEMA.ROW_ACCESS_POLICIES`;
"""

print("=== BigQuery Row-Level Security ===")
print(row_level_sql)
print("Key: If a user has NO policy granting access, they see ZERO rows.")

In [ ]:
# --- Column-Level Security with Policy Tags ---

column_security_sql = """
-- Step 1: Create a taxonomy in Data Catalog (via Console or API)
-- Taxonomy: "PII Classification"
--   Policy Tag: "Highly Sensitive" (SSN, credit card)
--   Policy Tag: "Sensitive" (email, phone)
--   Policy Tag: "Non-Sensitive" (name, city)

-- Step 2: Apply policy tags to columns
ALTER TABLE `project.dataset.customers`
ALTER COLUMN ssn SET POLICY TAG
  'projects/my-project/locations/us/taxonomies/123/policyTags/456';

ALTER TABLE `project.dataset.customers`  
ALTER COLUMN email SET POLICY TAG
  'projects/my-project/locations/us/taxonomies/123/policyTags/789';

-- Step 3: Grant access to specific groups
-- Role: roles/datacatalog.categoryFineGrainedReader
-- Granted on the policy tag to specific groups

-- Users WITHOUT the role see: <null> for tagged columns
-- Users WITH the role see: actual values
"""

print("=== Column-Level Security ===")
print(column_security_sql)

---
## 2. Encryption Options

Compare GMEK, CMEK, CSEK, and Cloud EKM encryption approaches.

In [ ]:
# --- Encryption Options Comparison ---

encryption = pd.DataFrame({
    'Option': ['GMEK (Default)', 'CMEK', 'CSEK', 'Cloud EKM'],
    'Key Management': [
        'Google manages everything',
        'You control keys in Cloud KMS',
        'You supply keys per API call',
        'Keys in external key manager'
    ],
    'Key Storage': [
        'Google infrastructure',
        'Cloud KMS (Google infrastructure)',
        'Your infrastructure (not stored by Google)',
        'External KMS (e.g., Thales, Fortanix)'
    ],
    'Supported Services': [
        'All GCP services',
        'BigQuery, GCS, Dataflow, Bigtable, Spanner, Dataproc',
        'Cloud Storage, Compute Engine disks only',
        'BigQuery, GCS, GKE, Compute Engine'
    ],
    'Use Case': [
        'Default, no compliance requirements',
        'Regulatory compliance, key rotation control',
        'Maximum control, keys never on Google',
        'Keys must never reside in Google infra'
    ]
})

print("=== Encryption Options Comparison ===")
for _, row in encryption.iterrows():
    print(f"\n{row['Option']}:")
    print(f"  Key Management: {row['Key Management']}")
    print(f"  Key Storage:    {row['Key Storage']}")
    print(f"  Services:       {row['Supported Services']}")
    print(f"  Use Case:       {row['Use Case']}")

In [ ]:
# --- CMEK Setup Commands ---

cmek_commands = """
# Step 1: Create a key ring and key in Cloud KMS
gcloud kms keyrings create my-ring --location=us
gcloud kms keys create my-key --keyring=my-ring --location=us \\
  --purpose=encryption --rotation-period=90d

# Step 2: Grant BigQuery service account access to the key
gcloud kms keys add-iam-policy-binding my-key \\
  --keyring=my-ring --location=us \\
  --member=serviceAccount:bq-PROJECT_NUMBER@bigquery-encryption.iam.gserviceaccount.com \\
  --role=roles/cloudkms.cryptoKeyEncrypterDecrypter

# Step 3: Create CMEK-protected BigQuery dataset
bq mk --dataset \\
  --default_kms_key=projects/my-proj/locations/us/keyRings/my-ring/cryptoKeys/my-key \\
  my-proj:secure_dataset

# Step 4: Create CMEK-protected Cloud Storage bucket
gsutil kms encryption \\
  -k projects/my-proj/locations/us/keyRings/my-ring/cryptoKeys/my-key \\
  gs://my-secure-bucket
"""

print("=== CMEK Setup Commands ===")
print(cmek_commands)

---
## 3. PII Detection and De-identification

Simulate Cloud DLP (Sensitive Data Protection) PII detection locally.

In [ ]:
import re
import hashlib

# --- Simulate PII Detection ---

sample_data = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['John Smith', 'Jane Doe', 'Bob Wilson', 'Alice Brown', 'Charlie Davis'],
    'email': ['john@example.com', 'jane.doe@company.org', 'bob@test.net', 'alice@mail.com', 'charlie@work.io'],
    'phone': ['(555) 123-4567', '555.987.6543', '+1-555-111-2222', '5553334444', '(555) 000-9999'],
    'ssn': ['123-45-6789', '987-65-4321', '111-22-3333', '444-55-6666', '777-88-9999'],
    'notes': ['Called about order #123', 'Email at jane.doe@company.org for refund',
             'SSN verified: 111-22-3333', 'Card ending 4242', 'No PII in this note']
})

# Simple PII detection patterns
patterns = {
    'EMAIL': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
    'PHONE': r'[\(]?\d{3}[\)]?[\s.-]?\d{3}[\s.-]?\d{4}',
    'SSN': r'\d{3}-\d{2}-\d{4}',
    'CREDIT_CARD_PARTIAL': r'\d{4}$',
}

print("=== PII Detection Results ===")
print(f"Scanning {len(sample_data)} records...\n")

for col in sample_data.columns:
    findings = []
    for pii_type, pattern in patterns.items():
        matches = sample_data[col].astype(str).str.contains(pattern, regex=True).sum()
        if matches > 0:
            findings.append(f"{pii_type}: {matches} matches")
    if findings:
        print(f"Column '{col}': {', '.join(findings)}")

print("\nOriginal data:")
sample_data

In [ ]:
# --- De-identification Techniques ---

def mask_value(value, show_last=4):
    """Replace characters with * except last N."""
    s = str(value)
    if len(s) <= show_last:
        return '*' * len(s)
    return '*' * (len(s) - show_last) + s[-show_last:]

def tokenize_value(value, key='secret_key'):
    """Replace with SHA-256 hash (deterministic, reversible with key)."""
    return hashlib.sha256(f"{key}:{value}".encode()).hexdigest()[:16]

def redact_pii_in_text(text):
    """Remove PII from free text."""
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[EMAIL_REDACTED]', text)
    text = re.sub(r'\d{3}-\d{2}-\d{4}', '[SSN_REDACTED]', text)
    text = re.sub(r'\d{4}$', '[CARD_REDACTED]', text)
    return text

# Apply de-identification
df_deidentified = sample_data.copy()
df_deidentified['email'] = df_deidentified['email'].apply(mask_value)
df_deidentified['phone'] = df_deidentified['phone'].apply(mask_value)
df_deidentified['ssn'] = df_deidentified['ssn'].apply(lambda x: tokenize_value(x))
df_deidentified['notes'] = df_deidentified['notes'].apply(redact_pii_in_text)

print("=== De-identified Data ===")
print("Techniques applied:")
print("  - email: Masking (show last 4 chars)")
print("  - phone: Masking (show last 4 chars)")
print("  - ssn: Tokenization (SHA-256 hash)")
print("  - notes: Redaction (replace PII with placeholders)")
print()
df_deidentified

---
## 4. Dataform SQLX Patterns

Show Dataform SQLX transformation examples and dependency management.

In [ ]:
# --- Dataform SQLX Examples ---

staging_sqlx = """
-- File: definitions/staging/stg_orders.sqlx
config {
  type: "view",
  schema: "staging",
  description: "Staged orders with quality filters and standardization",
  assertions: {
    uniqueKey: ["order_id"],
    nonNull: ["order_id", "customer_id", "order_date", "total_amount"]
  }
}

SELECT
  order_id,
  customer_id,
  PARSE_DATE('%Y-%m-%d', order_date) AS order_date,
  ROUND(total_amount, 2) AS total_amount,
  UPPER(TRIM(status)) AS status,
  LOWER(TRIM(region)) AS region,
  CURRENT_TIMESTAMP() AS _loaded_at
FROM ${ref("raw_orders")}
WHERE order_id IS NOT NULL
  AND total_amount > 0
  AND order_date IS NOT NULL
"""

incremental_sqlx = """
-- File: definitions/marts/fct_daily_revenue.sqlx
config {
  type: "incremental",
  schema: "marts",
  uniqueKey: ["order_date", "region"],
  bigquery: {
    partitionBy: "order_date",
    clusterBy: ["region"]
  }
}

SELECT
  order_date,
  region,
  COUNT(DISTINCT order_id) AS num_orders,
  COUNT(DISTINCT customer_id) AS num_customers,
  SUM(total_amount) AS total_revenue,
  AVG(total_amount) AS avg_order_value
FROM ${ref("stg_orders")}
${when(incremental(), `WHERE order_date > (SELECT MAX(order_date) FROM ${self()})`)}
GROUP BY 1, 2
"""

print("=== Dataform SQLX: Staging View ===")
print(staging_sqlx)
print("=== Dataform SQLX: Incremental Table ===")
print(incremental_sqlx)

---
## 5. Data Validation and Quality Checks

Demonstrate data quality validation patterns used in Dataplex and Dataform.

In [ ]:
import numpy as np

# --- Create sample data with quality issues ---
np.random.seed(42)
n = 500

df_quality = pd.DataFrame({
    'order_id': list(range(1, n+1)) + [1, 2, 3],  # 3 duplicates
    'customer_id': [f'C{i:04d}' for i in np.random.randint(1, 100, n)] + [None, 'C0001', 'C0002'],
    'order_date': list(pd.date_range('2025-01-01', periods=n, freq='h')) + 
                  [pd.Timestamp('2020-01-01'), pd.Timestamp('2025-02-01'), pd.Timestamp('2025-02-02')],
    'amount': list(np.random.lognormal(3, 1, n).round(2)) + [-50.0, 0.0, 1000000.0],
})

print(f"Dataset: {len(df_quality)} rows")
print(f"Intentional issues: 3 duplicate order_ids, 1 null customer_id,")
print(f"  1 old date, 1 negative amount, 1 extreme amount")
print()

# --- Data Quality Checks ---
checks = {
    'Uniqueness (order_id)': df_quality['order_id'].is_unique,
    'Completeness (customer_id not null)': df_quality['customer_id'].notna().mean(),
    'Validity (amount > 0)': (df_quality['amount'] > 0).mean(),
    'Validity (amount < 100000)': (df_quality['amount'] < 100000).mean(),
    'Freshness (data within 90 days)': (
        df_quality['order_date'] >= (pd.Timestamp.now() - pd.Timedelta(days=90))
    ).mean(),
    'Duplicate count': df_quality['order_id'].duplicated().sum(),
    'Null count (customer_id)': df_quality['customer_id'].isna().sum(),
}

print("=== Data Quality Check Results ===")
for check, result in checks.items():
    if isinstance(result, bool):
        status = 'PASS' if result else 'FAIL'
    elif isinstance(result, float):
        status = f'{result:.2%}' + (' PASS' if result >= 0.95 else ' FAIL')
    else:
        status = str(result) + (' PASS' if result == 0 else ' FAIL')
    print(f"  {check}: {status}")

In [ ]:
# --- Dataplex Data Quality YAML Spec ---

dataplex_yaml = """
# Dataplex Data Quality Task Configuration
rules:
  - column: order_id
    dimension: uniqueness
    threshold: 1.0
    description: "Order IDs must be unique"

  - column: customer_id
    dimension: completeness
    sql_expression: "customer_id IS NOT NULL"
    threshold: 1.0
    description: "Customer ID must not be null"

  - column: amount
    dimension: validity
    sql_expression: "amount > 0 AND amount < 1000000"
    threshold: 0.99
    description: "Amount must be positive and under 1M"

  - column: order_date
    dimension: freshness  
    sql_expression: "order_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)"
    threshold: 0.95
    description: "Data must be within last 90 days"

  - dimension: volume
    sql_expression: "COUNT(*) BETWEEN 400 AND 600"
    threshold: 1.0
    description: "Expected row count range"
"""

print("=== Dataplex Data Quality YAML ===")
print(dataplex_yaml)

---
## 6. Migration Service Comparison

Compare BigQuery Data Transfer Service, Database Migration Service, and Datastream.

In [ ]:
# --- Migration Services Comparison ---

migration_comparison = pd.DataFrame({
    'Service': [
        'BigQuery Data Transfer Service',
        'Database Migration Service (DMS)',
        'Datastream',
        'Transfer Appliance',
        'Storage Transfer Service'
    ],
    'Purpose': [
        'Import data into BigQuery from SaaS/cloud sources',
        'Migrate operational databases to GCP managed DBs',
        'Stream CDC changes to BigQuery/GCS for analytics',
        'Physical data transfer for large datasets',
        'Move data between cloud storage systems'
    ],
    'Sources': [
        'Google Ads, YouTube, S3, Redshift, Teradata',
        'MySQL, PostgreSQL, SQL Server, Oracle, AlloyDB',
        'Oracle, MySQL, PostgreSQL, AlloyDB',
        'On-premises storage (20+ TB)',
        'AWS S3, Azure Blob, HTTP/HTTPS, GCS'
    ],
    'Targets': [
        'BigQuery only',
        'Cloud SQL, AlloyDB, Spanner',
        'BigQuery, Cloud Storage',
        'Cloud Storage',
        'Cloud Storage'
    ],
    'Key Feature': [
        'Scheduled, managed, incremental loads',
        'Continuous replication, minimal downtime cutover',
        'Real-time CDC, serverless, log-based',
        'Offline transfer, encrypted, rack-mountable',
        'Scheduled, cross-cloud, bandwidth management'
    ]
})

print("=== Data Migration Services Comparison ===")
for _, row in migration_comparison.iterrows():
    print(f"\n{row['Service']}:")
    print(f"  Purpose: {row['Purpose']}")
    print(f"  Sources: {row['Sources']}")
    print(f"  Targets: {row['Targets']}")
    print(f"  Key:     {row['Key Feature']}")

In [ ]:
# --- Migration Decision Tree ---

scenarios = [
    ('Import Google Ads data into BigQuery', 'BigQuery Data Transfer Service'),
    ('Migrate on-prem MySQL to Cloud SQL', 'Database Migration Service'),
    ('Stream Oracle changes to BigQuery', 'Datastream'),
    ('Move 100TB from on-prem to GCS', 'Transfer Appliance'),
    ('Copy data from S3 to GCS on a schedule', 'Storage Transfer Service'),
    ('Migrate Teradata warehouse to BigQuery', 'BigQuery Data Transfer Service'),
    ('Real-time replication PostgreSQL to BQ', 'Datastream'),
    ('Migrate Oracle to AlloyDB', 'Database Migration Service'),
]

print("=== Migration Decision Scenarios ===")
for scenario, service in scenarios:
    print(f"  Scenario: {scenario}")
    print(f"  Answer:   {service}")
    print()

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **IAM** | Least privilege. Use predefined roles. Row-level + column-level security in BigQuery. |
| **Encryption** | Default = GMEK. Use CMEK for compliance. CSEK for GCS only. Cloud EKM for external keys. |
| **PII/DLP** | Inspection (discover), De-identification (mask/tokenize/redact), Risk analysis (k-anonymity). |
| **Dataform** | SQLX = SQL + config + refs. Assertions for quality. Incremental for efficiency. |
| **Data Validation** | Uniqueness, completeness, validity, freshness. Dataplex for automated quality tasks. |
| **Migration** | BQ DTS for SaaS imports. DMS for DB migration. Datastream for CDC to analytics. |